In [1]:
import os
gpu_num = 0 # Use "" to use the CPU
os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Colab preview compatibility.
try: # detect if the notebook runs in Colab
    import google.colab
    colab_compat = True # deactivate preview
except:
    colab_compat = False
resolution = [480,320] # increase for higher quality of renderings

# Allows to exit cell execution in Jupyter
class ExitCell(Exception):
    def _render_traceback_(self):
        pass

# Import Sionna
# try:
#     import sionna
# except ImportError as e:
#     # Install Sionna if package is not already installed
#     import os
#     os.system("pip install sionna")
#     import sionna

# Configure the notebook to use only a single GPU and allocate only as much memory as needed
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(gpus)
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)
# Avoid warnings from TensorFlow
tf.get_logger().setLevel('ERROR')

tf.random.set_seed(1) # Set global random seed for reproducibility

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import sys

from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, RadioMaterial, Camera
from sionna.rt.utils import r_hat
from sionna.constants import PI, SPEED_OF_LIGHT
from sionna.utils import expand_to_rank


[]


In [2]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import time

# Import Sionna RT components
from sionna.rt import load_scene, Transmitter, Receiver, PlanarArray, Camera

# For link-level simulations
from sionna.channel import cir_to_ofdm_channel, subcarrier_frequencies, OFDMChannel, ApplyOFDMChannel, CIRDataset
from sionna.nr import PUSCHConfig, PUSCHTransmitter, PUSCHReceiver
from sionna.utils import compute_ber, ebnodb2no, PlotBER
from sionna.ofdm import KBestDetector, LinearDetector
from sionna.mimo import StreamManagement

In [ ]:
# Load integrated scene
# sionna.rt.scene.munich
from pathlib import Path

def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "assets" / "raytrack_scene" / "scene.xml").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the RayLoc project root.")

PROJECT_ROOT = find_project_root()
GENERATED_DIR = PROJECT_ROOT / "generated"
GENERATED_DIR.mkdir(parents=True, exist_ok=True)
SCENE_PATH = PROJECT_ROOT / "assets" / "raytrack_scene" / "scene.xml"

scene = load_scene(str(SCENE_PATH))
scene.preview()

In [ ]:
scene.preview()

In [4]:
#convert lat lon to coordinate
def latlon_distance(lat, lon, lat_refer, lon_refer):
    radius = 6371

    lat = lat * np.pi / 180
    lon = lon * np.pi / 180
    lat_refer = lat_refer * np.pi / 180
    lon_refer = lon_refer * np.pi / 180
    
    #difference
    deltalat = lat - lat_refer
    deltalon = lon - lon_refer
    
    #x y difference
    x1 = deltalon * np.cos((lat + lat_refer) / 2)
    y1 = deltalat
    
    x = x1 * radius * 1000
    y = y1 * radius * 1000
    return (y, x)

In [5]:
#compute lat lon gap of each cell, depending on resolution

#original center: 47.655185, -122.307035
#window: 47.653750, -122.306494
#window coordinate: 40.5214, -159.5647

res = 1
origin_lat = 47.655185
window_lat = 47.653750
origin_lon = -122.307035
window_lon = -122.306494

v_distance = latlon_distance(window_lat, window_lon, origin_lat, origin_lon)[1]
h_distance = latlon_distance(window_lat, window_lon, origin_lat, origin_lon)[0]

#lat_gap
gap_lat = (window_lat - origin_lat) / (v_distance * res)
gap_lon = (window_lon - origin_lon) / (h_distance * res)

print(gap_lat)
print(gap_lon)

-3.541339253251836e-05
-3.3904737895661565e-06


In [ ]:
#calibration test
target = np.array([47.65074792715546, -122.312299126737])

index = (target - np.array([origin_lat, origin_lon])) / np.array([gap_lat, gap_lon])
print(index)


# set tx
scene.remove("tx")

tx = Transmitter(name="tx",
                 position=[100, 100, 0],
                 orientation=[0,0,0])

scene.add(tx)

scene.preview()

In [12]:
scene.remove("tx")
tx = Transmitter(name="tx",
                 position=[0, 0, 0],
                 orientation=[0,0,0])

scene.add(tx)

scene.preview()

Renderer(camera=PerspectiveCamera(aspect=1.31, children=(DirectionalLight(intensity=0.25, position=(0.0, 0.0, …

In [13]:
#configure building material
#def material itu concrete
# Sionna radio-material reference: https://nvlabs.github.io/sionna/api/rt.html#radio-materials
def my_material_callback(f_hz):
   a = 5.24
   b = 0
   c = 0.0462
   d = 0.7822

   relative_permittivity = a * f_hz**b
   conductivity = c * f_hz**d
   return (relative_permittivity, conductivity)

concrete_all_frequency = "custom_material_name1" 
custom_material = RadioMaterial(concrete_all_frequency,
                                frequency_update_callback=my_material_callback)

scene.add(custom_material)

In [18]:
#frequency, tx rx array, and material configuration
scene.frequency = 915e6 # in Hz; implicitly updates RadioMaterials

scene.synthetic_array = True # If set to False, ray tracing will be done per antenna element (slower for large arrays)

#print(scene.objects)
obj = scene.get(list(scene.objects.keys())[0])
print(list(scene.objects.keys())[0])
obj.radio_material = concrete_all_frequency
obj.radio_material.scattering_coefficient = 0.5

scene.tx_array = PlanarArray(num_rows=1,
                             num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern="dipole",
                             polarization="V")

scene.rx_array = scene.tx_array

ground


In [ ]:
# set tx

window_coordinate = latlon_distance(window_lat, window_lon, origin_lat, origin_lon)
#window_coordinate = latlon_distance(47.65471042652561, -122.30683509209548, origin_lat, origin_lon)

scene.remove("tx")

tx = Transmitter(name="tx",
                 position=[window_coordinate[0], window_coordinate[1], 10.6],
                 orientation=[0,0,0])

scene.add(tx)

In [16]:
#coverage map
cm = scene.coverage_map(max_depth=5, #less max_depth to see more contrast
                        diffraction=True, # Disable to see the effects of diffraction
                        cm_cell_size=(1., 1.), # Grid size of coverage map cells in m
                        combining_vec=None,
                        precoding_vec=None,
                        num_samples=int(1e6),
                        scattering = True,
                        edge_diffraction = True)

ValueError: Material 'itu_marble' is used by the object  'building_1' but is not well-defined.

In [ ]:
scene.preview()

In [ ]:
cm.show()

In [ ]:
#output coverage map
#convert to db
cm_df = pd.DataFrame(cm[0].numpy())
cm_db_df = 10 * np.log10(np.abs(cm_df))

coverage_output_dir = GENERATED_DIR / "sionna_power_map"
coverage_output_dir.mkdir(parents=True, exist_ok=True)
cm_db_df.to_csv(coverage_output_dir / "sionna_coverage_map_res1.csv", header=False, index=False)
#trim
# trimmed_df = cm_df.iloc[500:1000 + 1, 250:750 + 1]
# trimmed_df.to_csv("C:/Program Files/MATLAB/R2023a/bin/test/output_coverage/sionna_coverageMap_res1.csv", header=False, index = False)

In [ ]:
print(cm.rss)

In [ ]:
#simulate sig strength 1 by 1
import pandas as pd

height = 2
f = 915e6

sim_data_dir = GENERATED_DIR / "sionna_power_map"
sim_data_dir.mkdir(parents=True, exist_ok=True)

power_arr = np.zeros((cm.cell_centers.shape[0], cm.cell_centers.shape[1]))

for i in range(cm.cell_centers.shape[0]):
    for j in range(cm.cell_centers.shape[1]):
        # set tx
        a_arr = []
        tau_arr = []

        #set rx
        position = cm.cell_centers[i][j].numpy()
        x = position[0]
        y = position[1]

        scene.remove("rx")
        rx = Receiver(name="rx",
                position=[x, y, height],
                orientation=[0,0,0])
        scene.add(rx)

        paths = scene.compute_paths(max_depth=5,
                                    num_samples=1e6)
        paths.normalize_delays = False

        a, tau = paths.cir()
        a_arr = a[0, 0, 0, 0, 0, :, 0].numpy()
        tau_arr = tau[0, 0, 0, :].numpy()
        power = np.sum(a[0, 0, 0, 0, 0, :, 0].numpy() * np.exp(-1j * 2 * np.pi * f * tau[0, 0, 0, :].numpy()))
        print(20 * np.log10(abs(power)))

        power_arr[i, j] = 20 * np.log10(abs(power))
        with open(sim_data_dir / f"height{height}_amplitude.csv", "a+", newline="") as file:
            writer = csv.writer(file)
            writer.writerow([x, y, height] + [str(num) for num in a[0, 0, 0, 0, 0, :, 0].numpy()])
        with open(sim_data_dir / f"height{height}_delay.csv", "a+", newline="") as file:
            writer = csv.writer(file)
            writer.writerow([x, y, height] + [str(num) for num in tau[0, 0, 0, :].numpy()])
        #break
    power_data = pd.DataFrame(power_arr)
    power_data.to_csv(sim_data_dir / "sionna_coverage_map_pointwise_res1.csv", header=False, index=False)
#print(data.values[0])